# dbt on the lakehouse: getting started

[dbt](https://docs.getdbt.com/) turns SQL `select` statements into tables and views, tests them, and documents the dependency graph between them.
In this lab dbt talks to **Trino** (adapter `dbt-trino`), and Trino writes **Iceberg tables** to SeaweedFS through the Hive Metastore's Iceberg REST catalog, so everything dbt builds
is also readable from Spark.

This notebook walks through the demo project in `dbt/lakehouse_demo/`:
`seed` (load small CSV files) -> `run` (build models) -> `test` (check the results) -> `docs` (lineage).
It runs the same on Docker Compose and Kubernetes: the profile points at the host name `trino`.

In [ ]:
import sys, pathlib, json
import importlib.metadata as md
sys.path.append(str(pathlib.Path.cwd().parent))        # lab_utils.py lives one folder up
from lab_utils import dbt, trino, reset_dbt_demo, DBT_PROJECT

print("dbt-core", md.version("dbt-core"), "| dbt-trino", md.version("dbt-trino"))
print("project :", DBT_PROJECT)

## 1. The project
A dbt project is a folder of plain files. Print what is in it (ignoring dbt's generated `target/` and `logs/`):

In [ ]:
for p in sorted(DBT_PROJECT.rglob("*")):
    if p.is_file() and not {"target", "logs"} & set(p.relative_to(DBT_PROJECT).parts):
        print(p.relative_to(DBT_PROJECT))

In [ ]:
print((DBT_PROJECT / "profiles.yml").read_text())

`profiles.yml` is the connection: Trino host `trino`, catalog (`database`) `iceberg`, default schema `dbt_demo`. There is no authentication in the lab's Trino.

A **model** is one `select`. `ref()` declares a dependency, which is how dbt knows the build order:

In [ ]:
print((DBT_PROJECT / "models/staging/stg_orders.sql").read_text())
print("-" * 70)
print((DBT_PROJECT / "models/marts/customer_summary.sql").read_text())

## 2. Start from a clean slate and check the connection
`reset_dbt_demo()` drops the schemas this project creates (`dbt_demo`, `dbt_demo_raw`, `dbt_demo_snapshots`), so you can re-run the notebook any time.
`dbt debug` checks the profile and the connection to Trino.

In [ ]:
reset_dbt_demo()
_ = dbt("debug")

## 3. `dbt seed`: load the CSV files
Seeds are small CSV files in `seeds/` that dbt loads as tables. Here they stand in for raw data landed by an ingestion job (customers, products, orders).

In [ ]:
dbt("seed")

In [ ]:
trino("SELECT * FROM iceberg.dbt_demo_raw.raw_orders ORDER BY order_id LIMIT 5")

## 4. `dbt run`: build the models
Staging models are **views**; the marts are **tables** (`dbt_project.yml`). `fct_orders` is **incremental** (next notebook).
Because Trino's `iceberg` catalog is an Iceberg catalog, every dbt table is an Iceberg table; model configs can set Iceberg properties such as partitioning.

In [ ]:
dbt("run")

In [ ]:
print(trino("SHOW CREATE TABLE iceberg.dbt_demo.fct_orders").iloc[0, 0])

In [ ]:
trino("SELECT table_name, table_type FROM information_schema.tables WHERE table_schema = 'dbt_demo' ORDER BY table_name")

In [ ]:
trino("SELECT customer_id, name, plan, orders, round(total_spend, 2) AS total_spend, spend_tier FROM iceberg.dbt_demo.customer_summary ORDER BY total_spend DESC LIMIT 5")

## 5. `dbt test`: check the data and the logic
This runs the **data tests** declared in `schema.yml` (`unique`, `not_null`, `accepted_values`, `relationships`), the **singular test** in `tests/`, and the **unit test** for the spend-tier rule.
A test is a query that returns the *bad* rows; it passes when it returns none.

In [ ]:
dbt("test")

## 6. `dbt build`: everything in dependency order
`build` = seed + run + snapshot + test, executed in the order of the dependency graph. If a test on an upstream model fails, the models downstream of it are **skipped**,
so bad data does not flow into the marts (you will see that in notebook 02).

In [ ]:
dbt("build");

## 7. Documentation and lineage
`dbt docs generate` writes `target/manifest.json` (the project graph) and `target/catalog.json` (the real column types, read from Trino). `dbt docs serve` would browse them as a website; here we read the graph directly.

In [ ]:
dbt("docs", "generate")
manifest = json.loads((DBT_PROJECT / "target" / "manifest.json").read_text())
for node_id, parents in manifest["parent_map"].items():
    if node_id.startswith("model."):
        print(f"{node_id.split('.')[-1]:18s} <- {', '.join(p.split('.')[-1] for p in parents)}")

## Where to go next
* `02_incremental_and_snapshots`: incremental `MERGE` models, Iceberg time travel on dbt-built tables, slowly-changing dimensions.
* `03_tests_contracts_and_unit_tests`: failing tests on purpose, stored failures, model contracts, unit tests.
* The same tables are visible from Spark: `spark.table("iceberg_catalog.dbt_demo.fct_orders")`.